# 02 - EDA Storytelling Star Wars BI

Este notebook es el analisis exploratorio orientado al dashboard. Parte de los CSV limpios generados por `01_limpieza_star_wars.ipynb` y organiza la lectura segun el nuevo storytelling:

**Choose Your Side - Star Wars Rebellion Lab**

Objetivo: descubrir que experiencia Star Wars debe recibir cada tipo de audiencia para reactivar su conexion emocional con la marca.


## 0. Configuracion y carga de tablas limpias


In [5]:
from pathlib import Path
import pandas as pd
import numpy as np
import re


def find_project_root(start_path=None):
    start_path = Path(start_path or Path.cwd()).resolve()
    for candidate in [start_path, *start_path.parents]:
        if (candidate / "data" / "processed").exists() and (candidate / "notebooks").exists():
            return candidate
    return start_path

BASE_DIR = find_project_root()
PROCESSED_DIR = BASE_DIR / "data" / "processed"
DOCS_DIR = BASE_DIR / "docs"

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
print("Proyecto:", BASE_DIR)
print("Processed:", PROCESSED_DIR)


Proyecto: C:\Users\elena\OneDrive\Documentos\BOOTCAMP IA\DashboardStarWars
Processed: C:\Users\elena\OneDrive\Documentos\BOOTCAMP IA\DashboardStarWars\data\processed


In [6]:
# Tablas limpias generadas por 01_limpieza_star_wars.ipynb
survey_respondents = pd.read_csv(PROCESSED_DIR / "survey_respondents.csv")
survey_movies_seen = pd.read_csv(PROCESSED_DIR / "survey_movies_seen.csv")
survey_movie_rankings = pd.read_csv(PROCESSED_DIR / "survey_movie_rankings.csv")
survey_character_opinions = pd.read_csv(PROCESSED_DIR / "survey_character_opinions.csv")

universe_characters = pd.read_csv(PROCESSED_DIR / "universe_characters_clean.csv")
universe_films = pd.read_csv(PROCESSED_DIR / "universe_films_clean.csv")
universe_planets = pd.read_csv(PROCESSED_DIR / "universe_planets_clean.csv")
universe_starships = pd.read_csv(PROCESSED_DIR / "universe_starships_clean.csv")
universe_weapons = pd.read_csv(PROCESSED_DIR / "universe_weapons_clean.csv")
universe_quotes = pd.read_csv(PROCESSED_DIR / "universe_quotes_clean.csv")
universe_assets = pd.read_csv(PROCESSED_DIR / "universe_assets.csv")
universe_quality_summary = pd.read_csv(PROCESSED_DIR / "universe_quality_summary.csv")
films_business_clean = pd.read_csv(PROCESSED_DIR / "films_business_clean.csv", parse_dates=["release_date"])

print("Encuesta respondents:", survey_respondents.shape)
print("Opiniones personajes:", survey_character_opinions.shape)
print("Peliculas negocio:", films_business_clean.shape)
print("Activos universo:", universe_assets.shape)


Encuesta respondents: (1186, 23)
Opiniones personajes: (16604, 8)
Peliculas negocio: (11, 15)
Activos universo: (325, 10)


In [7]:
def missing_report(df):
    return (
        pd.DataFrame({
            "nulos": df.isna().sum(),
            "porcentaje": (df.isna().mean() * 100).round(2),
        })
        .query("nulos > 0")
        .sort_values("porcentaje", ascending=False)
    )


def split_keys(value):
    if pd.isna(value):
        return []
    return [item.strip() for item in str(value).split(",") if item.strip()]


def collect_list_keys(df, column):
    if column not in df.columns:
        return set()
    keys = set()
    for value in df[column].dropna():
        keys.update(split_keys(value))
    return keys


def export_csv(df, filename):
    path = PROCESSED_DIR / filename
    df.to_csv(path, index=False)
    print("Exportado:", filename, df.shape)
    return df


## Storytelling del informe

1. **La senal perdida:** localizar donde sigue viva la conexion con Star Wars.
2. **Los clanes de la galaxia:** segmentar la audiencia en tipos accionables.
3. **El mapa emocional de Star Wars:** interpretar personajes como emociones de marca.
4. **Las puertas de entrada al universo:** decidir que peliculas abren mejor la conversacion.
5. **Planetas como experiencias:** convertir mundos en atmosferas de campana.
6. **Tecnologia, poder y velocidad:** agrupar naves, vehiculos y armas como activos de espectaculo.
7. **La estrategia de reactivacion:** recomendar rutas de experiencia por tipo de publico.


In [8]:
# Episodio I: El mercado galactico
# Esta celda crea las tablas de la primera pagina del dashboard.
# Objetivo de negocio: medir si existe conexion activa con Star Wars y detectar si cambia por edad o genero.

# 1) KPIs generales de audiencia.
# Cada metrica resume la muestra completa y se usara como tarjeta en Power BI.
# Nota: mean() ignora los NaN, asi que los porcentajes se calculan sobre respuestas validas.
survey_kpis = pd.DataFrame([{
    "respondents": len(survey_respondents),
    "seen_any_star_wars_pct": round(survey_respondents["has_seen_any_star_wars_film_binary"].mean() * 100, 2),
    "star_wars_fan_pct": round(survey_respondents["is_star_wars_fan_binary"].mean() * 100, 2),
    "avg_movies_seen": round(survey_respondents["total_movies_seen"].mean(), 2),
    "complete_movie_ranking_pct": round(survey_respondents["has_complete_movie_ranking"].mean() * 100, 2),
    "with_demographic_info_pct": round(survey_respondents["has_demographic_info"].mean() * 100, 2),
}])

# 2) Fan rate por edad.
# Quitamos edades vacias para que cada barra represente un grupo real.
# respondents cuenta personas unicas; fan_rate_pct mide el porcentaje de fans dentro de cada edad.
fan_by_age = (
    survey_respondents.dropna(subset=["age"])
    .groupby("age", as_index=False)
    .agg(
        respondents=("respondent_id", "nunique"),
        fan_rate_pct=("is_star_wars_fan_binary", lambda s: round(s.mean() * 100, 2)),
        avg_movies_seen=("total_movies_seen", "mean"),
    )
)
fan_by_age["avg_movies_seen"] = fan_by_age["avg_movies_seen"].round(2)

# 3) Fan rate por genero.
# Misma logica que edad: compara conexion y consumo medio entre grupos con dato informado.
fan_by_gender = (
    survey_respondents.dropna(subset=["gender"])
    .groupby("gender", as_index=False)
    .agg(
        respondents=("respondent_id", "nunique"),
        fan_rate_pct=("is_star_wars_fan_binary", lambda s: round(s.mean() * 100, 2)),
        avg_movies_seen=("total_movies_seen", "mean"),
    )
)
fan_by_gender["avg_movies_seen"] = fan_by_gender["avg_movies_seen"].round(2)

# 4) Revision visual rapida dentro del notebook antes de exportar a CSV.
display(survey_kpis)
display(fan_by_age)
display(fan_by_gender)


,respondents,seen_any_star_wars_pct,star_wars_fan_pct,avg_movies_seen,complete_movie_ranking_pct,with_demographic_info_pct
0,1186,78.92,66.03,3.29,70.32,88.2


,age,respondents,fan_rate_pct,avg_movies_seen
0,18-29,218,68.89,4.24
1,30-44,268,72.46,3.94
2,45-60,291,64.17,3.66
3,> 60,269,58.55,2.90


,gender,respondents,fan_rate_pct,avg_movies_seen
0,female,549,59.95,3.10
1,male,497,71.63,4.27


In [9]:
# Episodio II: La pelicula que abre el portal
# Esta celda combina visionado, ranking y negocio para decidir que pelicula funciona mejor como puerta de entrada.

# 1) Alcance: porcentaje de personas que ha visto cada pelicula de la encuesta.
movie_views_summary = (
    survey_movies_seen
    .groupby(["episode_order", "film_key", "movie_title"], as_index=False)
    .agg(viewers=("has_seen_movie", "sum"), respondents=("respondent_id", "nunique"))
)
movie_views_summary["view_rate_pct"] = (movie_views_summary["viewers"] / movie_views_summary["respondents"] * 100).round(2)

# 2) Preferencia: ranking medio y primeros puestos entre quienes completaron el ranking.
movie_rank_summary = (
    survey_movie_rankings.dropna(subset=["movie_rank"])
    .groupby(["episode_order", "film_key", "movie_title"], as_index=False)
    .agg(
        avg_rank=("movie_rank", "mean"),
        median_rank=("movie_rank", "median"),
        ranking_responses=("respondent_id", "nunique"),
        first_place_votes=("movie_rank", lambda s: (s == 1).sum()),
    )
)
movie_rank_summary["avg_rank"] = movie_rank_summary["avg_rank"].round(2)
movie_rank_summary["first_place_pct"] = (movie_rank_summary["first_place_votes"] / movie_rank_summary["ranking_responses"] * 100).round(2)
movie_rank_summary["preference_score"] = (7 - movie_rank_summary["avg_rank"]).round(2)

# 3) Score compuesto: mezcla alcance, preferencia y primer puesto. No es prediccion de ventas.
movie_opportunities = movie_views_summary.merge(
    movie_rank_summary[["film_key", "avg_rank", "preference_score", "first_place_pct"]],
    on="film_key",
    how="left",
)
movie_opportunities["movie_campaign_score"] = (
    movie_opportunities["view_rate_pct"] * 0.45
    + (movie_opportunities["preference_score"] / 6 * 100) * 0.45
    + movie_opportunities["first_place_pct"] * 0.10
).round(2)
movie_opportunities = movie_opportunities.sort_values("movie_campaign_score", ascending=False)

movie_audience_summary = movie_views_summary.merge(
    movie_rank_summary[["film_key", "avg_rank", "preference_score", "ranking_responses", "first_place_pct"]],
    on="film_key",
    how="left",
)

# 4) Capa ejecutiva: cruza audiencia con taquilla/ROI para comparar conexion emocional y negocio.
eda_movie_commercial_audience_summary = films_business_clean.merge(movie_audience_summary, on="film_key", how="left")
eda_movie_commercial_audience_summary["movie_title"] = eda_movie_commercial_audience_summary["movie_title"].fillna(eda_movie_commercial_audience_summary["film_title"])
eda_movie_commercial_audience_summary["is_in_survey"] = eda_movie_commercial_audience_summary["viewers"].notna().astype(int)

best_campaign_movie = movie_opportunities.iloc[0]
best_box_office_movie = films_business_clean.sort_values("worldwide_box_office_usd", ascending=False).iloc[0]
best_roi_movie = films_business_clean.sort_values("roi", ascending=False).iloc[0]

display(movie_opportunities)
display(eda_movie_commercial_audience_summary)
print("Pelicula eje recomendada:", best_campaign_movie["movie_title"])
print("Mayor taquilla:", best_box_office_movie["film_title"])
print("Mayor ROI:", best_roi_movie["film_title"])


,episode_order,film_key,movie_title,viewers,respondents,view_rate_pct,avg_rank,preference_score,first_place_pct,movie_campaign_score
4,5,the_empire_strikes_back,Episode V: The Empire Strikes Back,758,1186,63.91,2.51,4.49,34.57,65.89
5,6,return_of_the_jedi,Episode VI: Return of the Jedi,738,1186,62.23,3.05,3.95,17.46,59.37
3,4,a_new_hope,Episode IV: A New Hope,607,1186,51.18,3.27,3.73,24.40,53.45
0,1,the_phantom_menace,Episode I: The Phantom Menace,673,1186,56.75,3.73,3.27,15.45,51.61
1,2,attack_of_the_clones,Episode II: Attack of the Clones,571,1186,48.15,4.09,2.91,3.83,43.88
2,3,revenge_of_the_sith,Episode III: Revenge of the Sith,550,1186,46.37,4.34,2.66,4.31,41.25


,film_key,film_title,the_numbers_title,release_date,release_year,era,film_type,budget_usd,domestic_box_office_usd,worldwide_box_office_usd,profit_estimated_usd,roi,data_status,source_name,source_url,episode_order,movie_title,viewers,respondents,view_rate_pct,avg_rank,preference_score,ranking_responses,first_place_pct,is_in_survey
0,the_phantom_menace,Episode I: The Phantom Menace,Star Wars: Episode I - The Phantom Menace,1999-05-19,1999,Prequel,Theatrical,115000000,431088295,1027044677,912044677,7.9308,Final,The Numbers,https://www.the-numbers.com/movies/franchise/S...,1.0,Episode I: The Phantom Menace,673.0,1186.0,56.75,3.73,3.27,835.0,15.45,1
1,attack_of_the_clones,Episode II: Attack of the Clones,Star Wars: Episode II - Attack of the Clones,2002-05-16,2002,Prequel,Theatrical,115000000,310676740,649398328,534398328,4.6469,Final,The Numbers,https://www.the-numbers.com/movies/franchise/S...,2.0,Episode II: Attack of the Clones,571.0,1186.0,48.15,4.09,2.91,836.0,3.83,1
2,revenge_of_the_sith,Episode III: Revenge of the Sith,Star Wars: Episode III - Revenge of the Sith,2005-05-19,2005,Prequel,Theatrical,113000000,380270577,848998877,735998877,6.5133,Final,The Numbers,https://www.the-numbers.com/movies/franchise/S...,3.0,Episode III: Revenge of the Sith,550.0,1186.0,46.37,4.34,2.66,835.0,4.31,1
3,a_new_hope,Episode IV: A New Hope,Star Wars: Episode IV - A New Hope,1977-05-25,1977,Original,Theatrical,11000000,460998007,775398007,764398007,69.4907,Final,The Numbers,https://www.the-numbers.com/movies/franchise/S...,4.0,Episode IV: A New Hope,607.0,1186.0,51.18,3.27,3.73,836.0,24.40,1
4,the_empire_strikes_back,Episode V: The Empire Strikes Back,Star Wars: Episode V - The Empire Strikes Back,1980-05-21,1980,Original,Theatrical,18000000,290475067,538375067,520375067,28.9097,Final,The Numbers,https://www.the-numbers.com/movies/franchise/S...,5.0,Episode V: The Empire Strikes Back,758.0,1186.0,63.91,2.51,4.49,836.0,34.57,1
5,return_of_the_jedi,Episode VI: Return of the Jedi,Star Wars: Episode VI - Return of the Jedi,1983-05-25,1983,Original,Theatrical,32500000,309306177,475106177,442606177,13.6187,Final,The Numbers,https://www.the-numbers.com/movies/franchise/S...,6.0,Episode VI: Return of the Jedi,738.0,1186.0,62.23,3.05,3.95,836.0,17.46,1
6,the_force_awakens,Episode VII: The Force Awakens,Star Wars: Episode VII - The Force Awakens,2015-12-18,2015,Sequel,Theatrical,245000000,936662225,2068223624,1823223624,7.4417,Final,The Numbers,https://www.the-numbers.com/movies/franchise/S...,NaN,Episode VII: The Force Awakens,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
7,the_last_jedi,Episode VIII: The Last Jedi,Star Wars: Episode VIII - The Last Jedi,2017-12-15,2017,Sequel,Theatrical,200000000,620181382,1332539889,1132539889,5.6627,Final,The Numbers,https://www.the-numbers.com/movies/franchise/S...,NaN,Episode VIII: The Last Jedi,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
8,the_rise_of_skywalker,Episode IX: The Rise of Skywalker,Star Wars: Episode IX - The Rise of Skywalker,2019-12-20,2019,Sequel,Theatrical,275000000,515202542,1074144248,799144248,2.9060,Final,The Numbers,https://www.the-numbers.com/movies/franchise/S...,NaN,Episode IX: The Rise of Skywalker,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
9,rogue_one,Rogue One: A Star Wars Story,Rogue One: A Star Wars Story,2016-12-16,2016,Sequel,Theatrical,200000000,532177324,1056257274,856257274,4.2813,Final,The Numbers,https://www.the-numbers.com/movies/franchise/S...,NaN,Rogue One: A Star Wars Story,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0


Pelicula eje recomendada: Episode V: The Empire Strikes Back
Mayor taquilla: Episode VII: The Force Awakens
Mayor ROI: Episode IV: A New Hope


In [10]:
# Episodio III: El consejo de personajes
# Esta celda resume opiniones de personajes y las cruza con presencia narrativa/calidad para priorizar activos de campana.

# 1) Afinidad de audiencia: favorabilidad, rechazo y desconocimiento por personaje.
character_opinion_summary = (
    survey_character_opinions
    .groupby(["character_key", "character_name"], as_index=False)
    .agg(
        avg_opinion_score=("opinion_score", "mean"),
        opinion_responses=("opinion_label", lambda s: s.notna().sum()),
        favorable_responses=("opinion_score", lambda s: (s > 0).sum()),
        unfavorable_responses=("opinion_score", lambda s: (s < 0).sum()),
        unfamiliar_responses=("is_unfamiliar", "sum"),
        total_rows=("respondent_id", "count"),
    )
)
character_opinion_summary["avg_opinion_score"] = character_opinion_summary["avg_opinion_score"].round(3)
character_opinion_summary["favorable_pct"] = (character_opinion_summary["favorable_responses"] / character_opinion_summary["opinion_responses"] * 100).round(2)
character_opinion_summary["unfavorable_pct"] = (character_opinion_summary["unfavorable_responses"] / character_opinion_summary["opinion_responses"] * 100).round(2)
character_opinion_summary["unfamiliar_pct"] = (character_opinion_summary["unfamiliar_responses"] / character_opinion_summary["opinion_responses"] * 100).round(2)
character_opinion_summary = character_opinion_summary.sort_values(["avg_opinion_score", "favorable_pct"], ascending=False)

# 2) Presencia narrativa: numero de frases disponibles por personaje.
quote_character_summary = (
    universe_quotes
    .groupby(["character_key", "character_name"], as_index=False)
    .agg(quote_count=("quote", "count"))
    .sort_values("quote_count", ascending=False)
)

characters_for_merge = universe_characters[["character_key", "name", "species", "gender", "homeworld", "height", "weight"]].rename(columns={"name": "universe_character_name"})
asset_quality_for_merge = universe_assets[universe_assets["asset_type"] == "character"][["asset_key", "data_completeness_pct", "internal_presence_score"]].rename(columns={"asset_key": "character_key"})

# 3) Enriquecimiento: unimos encuesta, universo y calidad de dato antes de calcular el indice.
character_opportunities = character_opinion_summary.merge(characters_for_merge, on="character_key", how="left")
character_opportunities = character_opportunities.merge(quote_character_summary[["character_key", "quote_count"]], on="character_key", how="left")
character_opportunities = character_opportunities.merge(asset_quality_for_merge, on="character_key", how="left")
character_opportunities["quote_count"] = character_opportunities["quote_count"].fillna(0)
character_opportunities["is_in_universe_dataset"] = character_opportunities["universe_character_name"].notna().astype(int)
character_opportunities["data_completeness_pct"] = character_opportunities["data_completeness_pct"].fillna(0)
character_opportunities["internal_presence_score"] = character_opportunities["internal_presence_score"].fillna(0) + character_opportunities["quote_count"]

character_opportunities["audience_affinity_score"] = (((character_opportunities["avg_opinion_score"] + 2) / 4) * 100).round(2)
character_opportunities["familiarity_score"] = (100 - character_opportunities["unfamiliar_pct"]).round(2)
max_presence = character_opportunities["internal_presence_score"].max()
character_opportunities["internal_presence_score_scaled"] = np.where(max_presence > 0, (character_opportunities["internal_presence_score"] / max_presence * 100).round(2), 0)
character_opportunities["data_quality_score"] = character_opportunities["data_completeness_pct"].round(2)
# 4) Indice direccional: pondera afinidad, familiaridad, presencia interna y calidad del dato.
character_opportunities["merchandising_potential_index"] = (
    character_opportunities["audience_affinity_score"] * 0.45
    + character_opportunities["familiarity_score"] * 0.25
    + character_opportunities["internal_presence_score_scaled"] * 0.20
    + character_opportunities["data_quality_score"] * 0.10
).round(2)

presence_median = character_opportunities["internal_presence_score_scaled"].median()
audience_median = character_opportunities["audience_affinity_score"].median()
character_opportunities["opportunity_quadrant"] = np.select(
    [
        (character_opportunities["internal_presence_score_scaled"] >= presence_median) & (character_opportunities["audience_affinity_score"] >= audience_median),
        (character_opportunities["internal_presence_score_scaled"] >= presence_median) & (character_opportunities["audience_affinity_score"] < audience_median),
        (character_opportunities["internal_presence_score_scaled"] < presence_median) & (character_opportunities["audience_affinity_score"] >= audience_median),
    ],
    ["priority_campaign", "repositioning_needed", "hidden_opportunity"],
    default="low_priority",
)
character_opportunities = character_opportunities.sort_values("merchandising_potential_index", ascending=False)

display(character_opportunities.head(14))


,character_key,character_name,avg_opinion_score,opinion_responses,favorable_responses,unfavorable_responses,unfamiliar_responses,total_rows,favorable_pct,unfavorable_pct,unfamiliar_pct,universe_character_name,species,gender,homeworld,height,weight,quote_count,data_completeness_pct,internal_presence_score,is_in_universe_dataset,audience_affinity_score,familiarity_score,internal_presence_score_scaled,data_quality_score,merchandising_potential_index,opportunity_quadrant
0,han_solo,Han Solo,1.672,829,761,9,15,1186,91.80,1.09,1.81,Han Solo,Human,Male,Corellia,1.80,80.0,17.0,100.00,22.0,1,91.80,98.19,100.00,100.00,95.86,priority_campaign
1,obi_wan_kenobi,Obi-Wan Kenobi,1.632,825,750,15,17,1186,90.91,1.82,2.06,Obi-Wan Kenobi,Human,Male,Stewjon,1.82,81.0,15.0,100.00,20.0,1,90.80,97.94,90.91,100.00,93.53,priority_campaign
2,yoda,Yoda,1.630,826,749,16,10,1186,90.68,1.94,1.21,Yoda,Yoda's species,Male,NaN,0.66,17.0,11.0,92.86,16.0,1,90.75,98.79,72.73,92.86,89.37,priority_campaign
3,luke_skywalker,Luke Skywalker,1.581,831,771,16,6,1186,92.78,1.93,0.72,Luke Skywalker,Human,Male,Tatooine,1.72,77.0,9.0,100.00,14.0,1,89.52,99.28,63.64,100.00,87.83,priority_campaign
5,leia_organa,Leia Organa,1.555,831,757,18,8,1186,91.10,2.17,0.96,Leia Organa,Human,Female,Alderaan,1.50,49.0,6.0,100.00,11.0,1,88.88,99.04,50.00,100.00,84.76,priority_campaign
10,darth_vader,Darth Vader,0.479,826,481,251,10,1186,58.23,30.39,1.21,Darth Vader,Human,Male,Tatooine,2.02,136.0,10.0,92.86,15.0,1,61.98,98.79,68.18,92.86,75.51,repositioning_needed
4,r2_d2,R2-D2,1.570,830,747,16,10,1186,90.00,1.93,1.20,R2-D2,Droid,NaN,Naboo,0.96,NaN,0.0,64.29,3.0,1,89.25,98.80,13.64,64.29,74.02,hidden_opportunity
6,c_3po,C-3PO,1.404,827,703,30,15,1186,85.01,3.63,1.81,C-3PO,Droid,NaN,Tatooine,1.71,NaN,2.0,64.29,5.0,1,85.10,98.19,22.73,64.29,73.82,hidden_opportunity
7,anakin_skywalker,Anakin Skywalker,0.776,823,514,122,52,1186,62.45,14.82,6.32,Anakin Skywalker,Human,Male,Tatooine,1.88,84.0,2.0,100.00,7.0,1,69.40,93.68,31.82,100.00,71.01,low_priority
9,padme_amidala,Padme Amidala,0.605,814,351,92,164,1186,43.12,11.30,20.15,Padme Amidala,Human,Female,Naboo,1.65,45.0,2.0,100.00,7.0,1,65.12,79.85,31.82,100.00,65.63,low_priority


In [11]:
# Paginas 5 y 6: Planetas, tecnologia y activos visuales
# Esta celda prepara activos de mundo, naves y armas. Ojo: Hoth y Dagobah son seleccion narrativa, no ranking cuantitativo.

universe_overview = pd.DataFrame([
    {"asset_type": "characters", "count": len(universe_characters)},
    {"asset_type": "films", "count": len(universe_films)},
    {"asset_type": "planets", "count": len(universe_planets)},
    {"asset_type": "starships", "count": len(universe_starships)},
    {"asset_type": "weapons", "count": len(universe_weapons)},
    {"asset_type": "quotes", "count": len(universe_quotes)},
])

character_species_summary = universe_characters.groupby("species", dropna=False, as_index=False).agg(characters=("id", "count")).sort_values("characters", ascending=False)
character_gender_summary = universe_characters.groupby("gender", dropna=False, as_index=False).agg(characters=("id", "count")).sort_values("characters", ascending=False)
character_homeworld_summary = universe_characters.groupby("homeworld", dropna=False, as_index=False).agg(characters=("id", "count")).sort_values("characters", ascending=False)

planet_business_summary = universe_planets[["planet_key", "name", "population", "diameter", "climate", "terrain", "resident_keys", "resident_count", "film_keys", "film_count"]].copy().sort_values(["film_count", "resident_count", "population"], ascending=False)
starship_business_summary = universe_starships[["starship_key", "name", "starship_class", "manufacturer", "cost_in_credits", "length", "crew", "passengers", "cargo_capacity", "pilot_keys", "pilot_count", "film_keys", "film_count"]].copy().sort_values(["film_count", "pilot_count", "cost_in_credits"], ascending=False)
weapon_business_summary = universe_weapons[["weapon_key", "name", "type", "manufacturer", "cost_in_credits", "length", "film_keys", "film_count"]].copy().sort_values(["film_count", "cost_in_credits"], ascending=False)

story_featured_assets = pd.DataFrame([
    {"story_line": "Planetas como experiencias", "asset_name": "Tatooine", "asset_type": "planet", "story_role": "aventura, origen, desierto y nostalgia", "experience_concept": "Pop-up retro o ruta de inicio"},
    {"story_line": "Planetas como experiencias", "asset_name": "Hoth", "asset_type": "planet", "story_role": "batalla, supervivencia y accion", "experience_concept": "Experiencia inmersiva de mision"},
    {"story_line": "Planetas como experiencias", "asset_name": "Dagobah", "asset_type": "planet", "story_role": "entrenamiento Jedi, misterio y sabiduria", "experience_concept": "Escape room o entrenamiento"},
    {"story_line": "Planetas como experiencias", "asset_name": "Coruscant", "asset_type": "planet", "story_role": "ciudad futurista, tecnologia y escala", "experience_concept": "Instalacion tecnologica"},
    {"story_line": "Planetas como experiencias", "asset_name": "Naboo", "asset_type": "planet", "story_role": "estetica visual, elegancia y lifestyle", "experience_concept": "Colaboracion lifestyle"},
    {"story_line": "Planetas como experiencias", "asset_name": "Endor", "asset_type": "planet", "story_role": "naturaleza, comunidad y aventura familiar", "experience_concept": "Evento familiar y exterior"},
    {"story_line": "Tecnologia, poder y velocidad", "asset_name": "Millennium Falcon", "asset_type": "starship", "story_role": "aventura, libertad y silueta reconocible", "experience_concept": "Pieza central fotografiable"},
    {"story_line": "Tecnologia, poder y velocidad", "asset_name": "X-wing", "asset_type": "starship", "story_role": "Alianza Rebelde, velocidad y coleccionismo", "experience_concept": "Gaming, maquetas y accion"},
    {"story_line": "Tecnologia, poder y velocidad", "asset_name": "Lightsaber", "asset_type": "weapon", "story_role": "simbolo transversal de eleccion y poder", "experience_concept": "Activacion Choose Your Side"},
])

strategy_planet_experiences = story_featured_assets.query("asset_type == 'planet'").rename(columns={"asset_name": "planet_name", "story_role": "brand_atmosphere"})[["planet_name", "brand_atmosphere", "experience_concept"]]

display(universe_overview)
display(story_featured_assets)


,asset_type,count
0,characters,112
1,films,11
2,planets,26
3,starships,56
4,weapons,57
5,quotes,89


,story_line,asset_name,asset_type,story_role,experience_concept
0,Planetas como experiencias,Tatooine,planet,"aventura, origen, desierto y nostalgia",Pop-up retro o ruta de inicio
1,Planetas como experiencias,Hoth,planet,"batalla, supervivencia y accion",Experiencia inmersiva de mision
2,Planetas como experiencias,Dagobah,planet,"entrenamiento Jedi, misterio y sabiduria",Escape room o entrenamiento
3,Planetas como experiencias,Coruscant,planet,"ciudad futurista, tecnologia y escala",Instalacion tecnologica
4,Planetas como experiencias,Naboo,planet,"estetica visual, elegancia y lifestyle",Colaboracion lifestyle
5,Planetas como experiencias,Endor,planet,"naturaleza, comunidad y aventura familiar",Evento familiar y exterior
6,"Tecnologia, poder y velocidad",Millennium Falcon,starship,"aventura, libertad y silueta reconocible",Pieza central fotografiable
7,"Tecnologia, poder y velocidad",X-wing,starship,"Alianza Rebelde, velocidad y coleccionismo","Gaming, maquetas y accion"
8,"Tecnologia, poder y velocidad",Lightsaber,weapon,simbolo transversal de eleccion y poder,Activacion Choose Your Side


In [12]:
# Paginas 2, 3 y 7: Segmentos, emociones y rutas finales
# Esta celda crea la capa estrategica: segmentos Choose Your Side, mapa emocional y rutas finales de activacion.

# Orden fijo para que Power BI muestre los segmentos como una historia y no alfabeticamente.
AUDIENCE_ORDER = {
    "Jedi fiel": 1,
    "Rebelde nostalgico": 2,
    "Explorador casual": 3,
    "Territorio neutral": 4,
}


# Reglas de negocio para convertir fan + consumo en los cuatro clanes de audiencia.
def classify_audience(row):
    is_fan = row.get("is_star_wars_fan_binary") == 1
    movies_seen = row.get("total_movies_seen", 0)
    if is_fan and movies_seen >= 5:
        return "Jedi fiel"
    if is_fan and movies_seen < 5:
        return "Rebelde nostalgico"
    if not is_fan and movies_seen >= 2:
        return "Explorador casual"
    return "Territorio neutral"


strategy_survey_respondents = survey_respondents.copy()
strategy_survey_respondents["audience_type"] = strategy_survey_respondents.apply(classify_audience, axis=1)
strategy_survey_respondents["audience_type_order"] = strategy_survey_respondents["audience_type"].map(AUDIENCE_ORDER)

total_respondents = len(strategy_survey_respondents)
strategy_audience_segments = (
    strategy_survey_respondents.groupby(["audience_type", "audience_type_order"], dropna=False)
    .agg(
        respondents=("respondent_id", "nunique"),
        avg_movies_seen=("total_movies_seen", "mean"),
        fan_rate_pct=("is_star_wars_fan_binary", "mean"),
        avg_character_affinity=("average_character_opinion_score", "mean"),
        with_demographic_info_pct=("has_demographic_info", "mean"),
    )
    .reset_index()
    .sort_values("audience_type_order")
)
strategy_audience_segments["share_pct"] = strategy_audience_segments["respondents"] / total_respondents * 100
strategy_audience_segments["fan_rate_pct"] = strategy_audience_segments["fan_rate_pct"] * 100
strategy_audience_segments["with_demographic_info_pct"] = strategy_audience_segments["with_demographic_info_pct"] * 100
strategy_audience_segments["strategic_role"] = strategy_audience_segments["audience_type"].map({
    "Jedi fiel": "Profundidad, lore, inmersion y pertenencia.",
    "Rebelde nostalgico": "Nostalgia, personajes clasicos y memoria emocional.",
    "Explorador casual": "Reconocimiento visual, iconos simples y acceso rapido.",
    "Territorio neutral": "Entrada ligera, digital y sin dependencia del lore.",
})
strategy_audience_segments["recommended_message"] = strategy_audience_segments["audience_type"].map({
    "Jedi fiel": "Entra en la mision completa.",
    "Rebelde nostalgico": "Vuelve al momento que encendio la saga.",
    "Explorador casual": "Reconoce los iconos y elige tu lado.",
    "Territorio neutral": "Descubre Star Wars por sus simbolos universales.",
})
strategy_audience_segments["activation_goal"] = strategy_audience_segments["audience_type"].map({
    "Jedi fiel": "Fidelizar y convertir en prescriptor.",
    "Rebelde nostalgico": "Reactivar recuerdo y compra emocional.",
    "Explorador casual": "Reducir friccion de entrada.",
    "Territorio neutral": "Crear primer contacto reconocible.",
})
strategy_audience_segments = strategy_audience_segments[[
    "audience_type_order", "audience_type", "respondents", "share_pct", "avg_movies_seen", "fan_rate_pct",
    "avg_character_affinity", "with_demographic_info_pct", "strategic_role", "recommended_message", "activation_goal"
]].round(2)

strategy_audience_age_matrix = (
    strategy_survey_respondents.groupby(["audience_type_order", "audience_type", "age"], dropna=False)
    .agg(respondents=("respondent_id", "nunique"), avg_movies_seen=("total_movies_seen", "mean"), fan_rate_pct=("is_star_wars_fan_binary", "mean"))
    .reset_index()
    .sort_values(["audience_type_order", "age"])
)
strategy_audience_age_matrix["fan_rate_pct"] = strategy_audience_age_matrix["fan_rate_pct"] * 100

# Diccionario narrativo: traduce cada personaje en una emocion de marca accionable.
emotion_map = {
    "luke_skywalker": ("Esperanza", "Heroe aspiracional", "Activaciones heroicas y mensajes de superacion."),
    "leia_organa": ("Liderazgo", "Autoridad rebelde", "Campanas de liderazgo, comunidad y resistencia."),
    "han_solo": ("Rebeldia", "Carisma inconformista", "Tono aventurero, nostalgico y de alta afinidad."),
    "yoda": ("Sabiduria", "Mentor Jedi", "Experiencias de aprendizaje, misterio y entrenamiento."),
    "obi_wan_kenobi": ("Legado", "Puente generacional", "Narrativa clasica con autoridad y memoria."),
    "darth_vader": ("Poder", "Icono premium polarizante", "Linea visual adulta, intensa y de alto reconocimiento."),
    "anakin_skywalker": ("Conflicto", "Transformacion", "Relatos de dualidad y eleccion de bando."),
    "r2_d2": ("Compania", "Humor y ternura", "Entrada amable para publico familiar y casual."),
    "c_3po": ("Humor", "Compania reconocible", "Activaciones ligeras, familiares y nostalgicas."),
    "emperor_palpatine": ("Amenaza", "Riesgo dramatico", "Contrapunto del Lado Oscuro, uso secundario."),
    "jar_jar_binks": ("Riesgo de rechazo", "Personaje polarizante", "Usar solo como aprendizaje de sesgo y tono."),
    "boba_fett": ("Misterio", "Coleccionismo", "Linea nicho para fans de iconografia y armaduras."),
    "padme_amidala": ("Elegancia", "Politica y estilo", "Activaciones lifestyle y estetica Naboo."),
    "lando_calrissian": ("Estilo", "Carisma secundario", "Campanas de nostalgia, moda y diferenciacion."),
}

strategy_character_emotional_map = character_opportunities.copy()
strategy_character_emotional_map["brand_emotion"] = strategy_character_emotional_map["character_key"].map(lambda key: emotion_map.get(key, ("Afinidad", "Activo de conexion", "Apoyo narrativo segun segmento."))[0])
strategy_character_emotional_map["emotional_role"] = strategy_character_emotional_map["character_key"].map(lambda key: emotion_map.get(key, ("Afinidad", "Activo de conexion", "Apoyo narrativo segun segmento."))[1])
strategy_character_emotional_map["activation_use"] = strategy_character_emotional_map["character_key"].map(lambda key: emotion_map.get(key, ("Afinidad", "Activo de conexion", "Apoyo narrativo segun segmento."))[2])
strategy_character_emotional_map["polarization_risk"] = strategy_character_emotional_map["unfavorable_pct"].apply(lambda value: "alto" if value >= 25 else "medio" if value >= 10 else "bajo")
strategy_character_emotional_map = strategy_character_emotional_map[[
    "character_key", "character_name", "brand_emotion", "emotional_role", "activation_use", "familiarity_score",
    "audience_affinity_score", "quote_count", "favorable_pct", "unfavorable_pct", "opinion_responses",
    "opportunity_quadrant", "polarization_risk"
]].sort_values(["audience_affinity_score", "familiarity_score"], ascending=False)

# Rutas finales: una recomendacion concreta para cada segmento.
strategy_experience_routes = pd.DataFrame([
    {"audience_type_order": 1, "audience_type": "Jedi fiel", "entry_gate": "The Empire Strikes Back", "key_characters": "Yoda, Luke Skywalker, Darth Vader", "world": "Dagobah / Hoth", "experience_format": "Experiencia inmersiva y mision por niveles", "primary_emotion": "Profundidad y pertenencia", "recommended_action": "Activar lore, retos, coleccionismo y contenido desbloqueable."},
    {"audience_type_order": 2, "audience_type": "Rebelde nostalgico", "entry_gate": "Trilogia original", "key_characters": "Han Solo, Leia Organa, Obi-Wan Kenobi", "world": "Tatooine", "experience_format": "Campana emocional retro", "primary_emotion": "Nostalgia y rebeldia", "recommended_action": "Usar escenas, frases y estetica clasica con baja complejidad."},
    {"audience_type_order": 3, "audience_type": "Explorador casual", "entry_gate": "Iconos reconocibles", "key_characters": "Darth Vader, Yoda, R2-D2", "world": "Escenarios reconocibles", "experience_format": "Activacion visual sencilla", "primary_emotion": "Reconocimiento inmediato", "recommended_action": "Priorizar simbolos, videos cortos, piezas sociales y rutas simples."},
    {"audience_type_order": 4, "audience_type": "Territorio neutral", "entry_gate": "Lightsaber y naves", "key_characters": "Simbolos antes que personajes", "world": "Universo simplificado", "experience_format": "Activacion digital de entrada", "primary_emotion": "Curiosidad", "recommended_action": "Evitar exceso de lore y crear una primera interaccion rapida."},
])

campaign_lines = strategy_experience_routes.rename(columns={
    "audience_type": "campaign_line",
    "key_characters": "main_assets",
    "experience_format": "recommended_products",
    "recommended_action": "business_reading",
}).copy()
campaign_lines["target"] = campaign_lines["campaign_line"]
campaign_lines = campaign_lines[["campaign_line", "main_assets", "recommended_products", "target", "business_reading", "entry_gate", "world", "primary_emotion"]]

storytelling_pages = pd.DataFrame([
    {"page_order": 1, "page_name": "La senal perdida", "business_question": "Donde sigue viva la conexion con Star Wars?", "main_tables": "eda_survey_kpis, eda_fan_by_age, eda_fan_by_gender, survey_respondents", "main_visuals": "KPIs, barras por edad/genero, segmentador fan_segment"},
    {"page_order": 2, "page_name": "Los clanes de la galaxia", "business_question": "Que tipos de audiencia necesita reactivar la marca?", "main_tables": "strategy_audience_segments, strategy_audience_age_matrix, strategy_survey_respondents", "main_visuals": "donut o barras de segmentos, matriz segmento x edad, promedio peliculas vistas"},
    {"page_order": 3, "page_name": "El mapa emocional de Star Wars", "business_question": "Que emocion de marca activa cada personaje?", "main_tables": "strategy_character_emotional_map, eda_character_merchandising_opportunities, eda_quote_character_summary", "main_visuals": "dispersion afinidad/familiaridad, amor vs rechazo, frases por personaje"},
    {"page_order": 4, "page_name": "Las puertas de entrada al universo", "business_question": "Que pelicula abre mejor la conversacion con cada publico?", "main_tables": "eda_movie_opportunities, eda_movie_commercial_audience_summary, films_business_clean", "main_visuals": "score pelicula, popularidad vs preferencia, impacto comercial vs conexion"},
    {"page_order": 5, "page_name": "Planetas como experiencias", "business_question": "Que atmosfera debe vivir cada publico?", "main_tables": "strategy_planet_experiences, eda_planet_business_summary, universe_planets_clean", "main_visuals": "tarjetas de experiencia, film_count, resident_count, mapa de mundos"},
    {"page_order": 6, "page_name": "Tecnologia, poder y velocidad", "business_question": "Que activos generan impacto visual y accion?", "main_tables": "eda_starship_business_summary, eda_weapon_business_summary, universe_starships_clean, universe_weapons_clean", "main_visuals": "naves por presencia, clase de nave, coste vs presencia, armas iconicas"},
    {"page_order": 7, "page_name": "La estrategia de reactivacion", "business_question": "Como convertir los datos en rutas de experiencia?", "main_tables": "strategy_experience_routes, story_campaign_lines, eda_conclusions, eda_survey_bias_visual", "main_visuals": "matriz final por audiencia, tarjetas de ruta, sesgos clave"},
])

display(strategy_audience_segments)
display(strategy_character_emotional_map.head(10))
display(strategy_experience_routes)


,audience_type_order,audience_type,respondents,share_pct,avg_movies_seen,fan_rate_pct,avg_character_affinity,with_demographic_info_pct,strategic_role,recommended_message,activation_goal
1,1,Jedi fiel,443,37.35,5.93,100.0,1.10,97.97,"Profundidad, lore, inmersion y pertenencia.",Entra en la mision completa.,Fidelizar y convertir en prescriptor.
2,2,Rebelde nostalgico,109,9.19,3.17,100.0,1.18,98.17,"Nostalgia, personajes clasicos y memoria emoci...",Vuelve al momento que encendio la saga.,Reactivar recuerdo y compra emocional.
0,3,Explorador casual,234,19.73,3.75,0.0,0.92,98.29,"Reconocimiento visual, iconos simples y acceso...",Reconoce los iconos y elige tu lado.,Reducir friccion de entrada.
3,4,Territorio neutral,400,33.73,0.12,0.0,0.72,68.75,"Entrada ligera, digital y sin dependencia del ...",Descubre Star Wars por sus simbolos universales.,Crear primer contacto reconocible.


,character_key,character_name,brand_emotion,emotional_role,activation_use,familiarity_score,audience_affinity_score,quote_count,favorable_pct,unfavorable_pct,opinion_responses,opportunity_quadrant,polarization_risk
0,han_solo,Han Solo,Rebeldia,Carisma inconformista,"Tono aventurero, nostalgico y de alta afinidad.",98.19,91.80,17.0,91.80,1.09,829,priority_campaign,bajo
1,obi_wan_kenobi,Obi-Wan Kenobi,Legado,Puente generacional,Narrativa clasica con autoridad y memoria.,97.94,90.80,15.0,90.91,1.82,825,priority_campaign,bajo
2,yoda,Yoda,Sabiduria,Mentor Jedi,"Experiencias de aprendizaje, misterio y entren...",98.79,90.75,11.0,90.68,1.94,826,priority_campaign,bajo
3,luke_skywalker,Luke Skywalker,Esperanza,Heroe aspiracional,Activaciones heroicas y mensajes de superacion.,99.28,89.52,9.0,92.78,1.93,831,priority_campaign,bajo
4,r2_d2,R2-D2,Compania,Humor y ternura,Entrada amable para publico familiar y casual.,98.80,89.25,0.0,90.00,1.93,830,hidden_opportunity,bajo
5,leia_organa,Leia Organa,Liderazgo,Autoridad rebelde,"Campanas de liderazgo, comunidad y resistencia.",99.04,88.88,6.0,91.10,2.17,831,priority_campaign,bajo
6,c_3po,C-3PO,Humor,Compania reconocible,"Activaciones ligeras, familiares y nostalgicas.",98.19,85.10,2.0,85.01,3.63,827,hidden_opportunity,bajo
7,anakin_skywalker,Anakin Skywalker,Conflicto,Transformacion,Relatos de dualidad y eleccion de bando.,93.68,69.40,2.0,62.45,14.82,823,low_priority,medio
8,lando_calrissian,Lando Calrissian,Estilo,Carisma secundario,"Campanas de nostalgia, moda y diferenciacion.",81.95,65.92,2.0,44.51,8.66,820,low_priority,bajo
9,padme_amidala,Padme Amidala,Elegancia,Politica y estilo,Activaciones lifestyle y estetica Naboo.,79.85,65.12,2.0,43.12,11.30,814,low_priority,medio


,audience_type_order,audience_type,entry_gate,key_characters,world,experience_format,primary_emotion,recommended_action
0,1,Jedi fiel,The Empire Strikes Back,"Yoda, Luke Skywalker, Darth Vader",Dagobah / Hoth,Experiencia inmersiva y mision por niveles,Profundidad y pertenencia,"Activar lore, retos, coleccionismo y contenido..."
1,2,Rebelde nostalgico,Trilogia original,"Han Solo, Leia Organa, Obi-Wan Kenobi",Tatooine,Campana emocional retro,Nostalgia y rebeldia,"Usar escenas, frases y estetica clasica con ba..."
2,3,Explorador casual,Iconos reconocibles,"Darth Vader, Yoda, R2-D2",Escenarios reconocibles,Activacion visual sencilla,Reconocimiento inmediato,"Priorizar simbolos, videos cortos, piezas soci..."
3,4,Territorio neutral,Lightsaber y naves,Simbolos antes que personajes,Universo simplificado,Activacion digital de entrada,Curiosidad,Evitar exceso de lore y crear una primera inte...


In [13]:
# Calidad, sesgos y checks de relaciones
# Esta celda separa dos cosas: sesgos para explicar la lectura del dashboard y checks tecnicos de claves entre tablas.

# Ranking de nulos: sirve para explicar limites del dato antes de presentar conclusiones.
survey_missing_top = missing_report(survey_respondents).reset_index().rename(columns={"index": "column"})
survey_missing_top["dataset"] = "survey_respondents"

universe_missing_long = []
for dataset_name, df in {
    "characters": universe_characters,
    "films": universe_films,
    "planets": universe_planets,
    "starships": universe_starships,
    "weapons": universe_weapons,
    "quotes": universe_quotes,
}.items():
    temp = missing_report(df).reset_index().rename(columns={"index": "column"})
    temp["dataset"] = dataset_name
    universe_missing_long.append(temp)

universe_missing_long = pd.concat(universe_missing_long, ignore_index=True) if universe_missing_long else pd.DataFrame(columns=["column", "nulos", "porcentaje", "dataset"])
governance_missing_top = pd.concat([survey_missing_top[["dataset", "column", "nulos", "porcentaje"]], universe_missing_long[["dataset", "column", "nulos", "porcentaje"]]], ignore_index=True).sort_values(["porcentaje", "nulos"], ascending=False)

# Sesgos principales que deben aparecer en la lectura critica del dashboard.
survey_sample_bias = pd.DataFrame([
    {"risk": "Muestra orientada a personas que conocen Star Wars", "evidence": f"{survey_kpis.loc[0, 'seen_any_star_wars_pct']}% declara haber visto alguna pelicula.", "business_impact": "Puede sobreestimar la demanda real del publico general."},
    {"risk": "Sesgo fan", "evidence": f"{survey_kpis.loc[0, 'star_wars_fan_pct']}% se declara fan entre respuestas validas.", "business_impact": "Las preferencias pueden favorecer personajes iconicos frente a nichos de crecimiento."},
    {"risk": "Datos demograficos incompletos", "evidence": f"{survey_kpis.loc[0, 'with_demographic_info_pct']}% tiene alguna informacion demografica util.", "business_impact": "La segmentacion por edad, genero, ingresos o region debe interpretarse con cautela."},
    {"risk": "Contenido expandido sin dimension propia", "evidence": "Algunas listas contienen series, videojuegos o personajes secundarios sin tabla maestra.", "business_impact": "No debe forzarse una relacion si el dashboard no analiza ese universo expandido."},
])

# Checks tecnicos: fail = clave rota que bloquea; warn = valor fuera de dimension pero aceptable como limitacion.
quality_checks = []
def add_quality_check(check_name, status, affected_rows, detail):
    quality_checks.append({"check_name": check_name, "status": status, "affected_rows": int(affected_rows), "detail": detail})

survey_character_keys = set(survey_character_opinions["character_key"].dropna().unique())
universe_character_keys = set(universe_characters["character_key"].dropna().unique())
survey_film_keys = set(survey_movies_seen["film_key"].dropna().unique()) | set(survey_movie_rankings["film_key"].dropna().unique())
universe_film_keys = set(universe_films["film_key"].dropna().unique())
business_film_keys = set(films_business_clean["film_key"].dropna().unique())

for name, missing in [
    ("survey_characters_exist_in_universe", sorted(survey_character_keys - universe_character_keys)),
    ("survey_films_exist_in_universe_films", sorted(survey_film_keys - universe_film_keys)),
    ("survey_films_exist_in_business_films", sorted(survey_film_keys - business_film_keys)),
]:
    add_quality_check(name, "pass" if not missing else "fail", len(missing), ", ".join(missing) if missing else "OK")

for check_name, df, column, allowed_keys in [
    ("planet_resident_keys_in_characters", universe_planets, "resident_keys", universe_character_keys),
    ("starship_pilot_keys_in_characters", universe_starships, "pilot_keys", universe_character_keys),
    ("vehicle_pilot_keys_in_characters", pd.read_csv(PROCESSED_DIR / "universe_vehicles_clean.csv"), "pilot_keys", universe_character_keys),
    ("planet_film_keys_in_films", universe_planets, "film_keys", universe_film_keys | {"all_episodes"}),
    ("starship_film_keys_in_films", universe_starships, "film_keys", universe_film_keys | {"all_episodes"}),
    ("weapon_film_keys_in_films", universe_weapons, "film_keys", universe_film_keys | {"all_episodes"}),
]:
    missing = sorted(collect_list_keys(df, column) - allowed_keys)
    add_quality_check(check_name, "pass" if not missing else "warn", len(missing), ", ".join(missing[:30]) if missing else "OK")

relationship_quality_checks = pd.DataFrame(quality_checks)
display(governance_missing_top.head(20))
display(survey_sample_bias)
display(relationship_quality_checks)


,dataset,column,nulos,porcentaje
0,survey_respondents,is_expanded_universe_fan,973,82.04
1,survey_respondents,is_expanded_universe_fan_binary,973,82.04
32,starships,cost_in_credits,40,71.43
25,planets,diameter,15,57.69
26,planets,population,15,57.69
15,characters,year_died,62,55.36
27,planets,rotation_period,14,53.85
28,planets,orbital_period,14,53.85
29,planets,surface_water,14,53.85
43,weapons,cost_in_credits,25,43.86


,risk,evidence,business_impact
0,Muestra orientada a personas que conocen Star ...,78.92% declara haber visto alguna pelicula.,Puede sobreestimar la demanda real del publico...
1,Sesgo fan,66.03% se declara fan entre respuestas validas.,Las preferencias pueden favorecer personajes i...
2,Datos demograficos incompletos,88.2% tiene alguna informacion demografica util.,"La segmentacion por edad, genero, ingresos o r..."
3,Contenido expandido sin dimension propia,"Algunas listas contienen series, videojuegos o...",No debe forzarse una relacion si el dashboard ...


,check_name,status,affected_rows,detail
0,survey_characters_exist_in_universe,pass,0,OK
1,survey_films_exist_in_universe_films,pass,0,OK
2,survey_films_exist_in_business_films,pass,0,OK
3,planet_resident_keys_in_characters,warn,1,poggle_the_lesser
4,starship_pilot_keys_in_characters,warn,5,"clone_force_99, darth_maul, dash_rendar, gener..."
5,vehicle_pilot_keys_in_characters,warn,2,"darth_maul, scout_trooper"
6,planet_film_keys_in_films,warn,6,"ahsoka, andor, obi_wan_kenobi, star_wars_rebel..."
7,starship_film_keys_in_films,warn,7,"ahsoka, andor, shadows_of_the_empire, star_war..."
8,weapon_film_keys_in_films,warn,7,"star_wars_battlefront, star_wars_galaxy_s_edge..."


In [14]:
# Conclusiones ejecutivas
# Esta celda convierte los hallazgos principales en frases ejecutivas listas para Power BI.

best_movie_by_rank = eda_movie_commercial_audience_summary.sort_values("preference_score", ascending=False).iloc[0]
top_segment = strategy_audience_segments.sort_values("respondents", ascending=False).iloc[0]
top_character_emotion = strategy_character_emotional_map.sort_values("audience_affinity_score", ascending=False).iloc[0]

eda_conclusions = pd.DataFrame([
    {"area": "Audiencia", "finding": f"{survey_kpis.iloc[0]['seen_any_star_wars_pct']:.2f}% ha visto alguna pelicula y {survey_kpis.iloc[0]['star_wars_fan_pct']:.2f}% se declara fan.", "business_reading": "La marca tiene reconocimiento, pero la estrategia debe diferenciar niveles de vinculo."},
    {"area": "Segmentacion", "finding": f"El segmento mas grande es {top_segment['audience_type']} con {int(top_segment['respondents'])} respuestas.", "business_reading": "El dashboard debe hablar de clanes de audiencia, no de una masa unica."},
    {"area": "Peliculas", "finding": f"{best_movie_by_rank['movie_title']} lidera la conexion de audiencia con preference_score {best_movie_by_rank['preference_score']:.2f}.", "business_reading": "La pelicula ganadora funciona como puerta emocional, no solo como dato comercial."},
    {"area": "Personajes", "finding": f"{top_character_emotion['character_name']} destaca por afinidad y activa la emocion {top_character_emotion['brand_emotion']}.", "business_reading": "Los personajes deben leerse como emociones de marca, no solo como productos."},
    {"area": "Experiencias", "finding": "Tatooine, Hoth, Dagobah, Coruscant, Naboo y Endor representan atmosferas de campana distintas.", "business_reading": "Elegir un planeta equivale a elegir la sensacion que vivira el publico."},
    {"area": "Activos visuales", "finding": "Millennium Falcon, X-wing y Lightsaber concentran reconocimiento visual y accion.", "business_reading": "Los simbolos convierten la estrategia en una experiencia memorable y facil de comunicar."},
    {"area": "Recomendacion", "finding": "La propuesta final es Choose Your Side: una estrategia modular por tipo de audiencia.", "business_reading": "Star Wars no necesita una unica campana; necesita varias rutas de conexion bajo una misma marca."},
    {"area": "Sesgos", "finding": "La encuesta esta inclinada hacia personas familiarizadas con Star Wars.", "business_reading": "Las conclusiones son direccionales y no deben venderse como prediccion exacta del publico general."},
])

display(eda_conclusions)


,area,finding,business_reading
0,Audiencia,78.92% ha visto alguna pelicula y 66.03% se de...,"La marca tiene reconocimiento, pero la estrate..."
1,Segmentacion,El segmento mas grande es Jedi fiel con 443 re...,El dashboard debe hablar de clanes de audienci...
2,Peliculas,Episode V: The Empire Strikes Back lidera la c...,La pelicula ganadora funciona como puerta emoc...
3,Personajes,Han Solo destaca por afinidad y activa la emoc...,Los personajes deben leerse como emociones de ...
4,Experiencias,"Tatooine, Hoth, Dagobah, Coruscant, Naboo y En...",Elegir un planeta equivale a elegir la sensaci...
5,Activos visuales,"Millennium Falcon, X-wing y Lightsaber concent...",Los simbolos convierten la estrategia en una e...
6,Recomendacion,La propuesta final es Choose Your Side: una es...,Star Wars no necesita una unica campana; neces...
7,Sesgos,La encuesta esta inclinada hacia personas fami...,Las conclusiones son direccionales y no deben ...


In [15]:
# Exportacion de resultados EDA y storytelling
# Esta celda guarda todas las tablas finales en data/processed para que el workbook de Power BI pueda importarlas.

exports = {
    "eda_survey_kpis.csv": survey_kpis,
    "eda_fan_by_age.csv": fan_by_age,
    "eda_fan_by_gender.csv": fan_by_gender,
    "eda_movie_views_summary.csv": movie_views_summary,
    "eda_movie_rank_summary.csv": movie_rank_summary,
    "eda_movie_opportunities.csv": movie_opportunities,
    "eda_movie_commercial_audience_summary.csv": eda_movie_commercial_audience_summary,
    "eda_character_opinion_summary.csv": character_opinion_summary,
    "eda_quote_character_summary.csv": quote_character_summary,
    "eda_character_merchandising_opportunities.csv": character_opportunities,
    "eda_universe_overview.csv": universe_overview,
    "eda_character_species_summary.csv": character_species_summary,
    "eda_character_gender_summary.csv": character_gender_summary,
    "eda_character_homeworld_summary.csv": character_homeworld_summary,
    "eda_planet_business_summary.csv": planet_business_summary,
    "eda_starship_business_summary.csv": starship_business_summary,
    "eda_weapon_business_summary.csv": weapon_business_summary,
    "eda_governance_missing_top.csv": governance_missing_top,
    "eda_survey_sample_bias.csv": survey_sample_bias,
    "eda_relationship_quality_checks.csv": relationship_quality_checks,
    "eda_conclusions.csv": eda_conclusions,
    "story_featured_assets.csv": story_featured_assets,
    "story_campaign_lines.csv": campaign_lines,
    "storytelling_powerbi_pages.csv": storytelling_pages,
    "strategy_audience_segments.csv": strategy_audience_segments,
    "strategy_audience_age_matrix.csv": strategy_audience_age_matrix,
    "strategy_survey_respondents.csv": strategy_survey_respondents,
    "strategy_character_emotional_map.csv": strategy_character_emotional_map,
    "strategy_planet_experiences.csv": strategy_planet_experiences,
    "strategy_experience_routes.csv": strategy_experience_routes,
}

for filename, df in exports.items():
    export_csv(df, filename)

failed_checks = relationship_quality_checks.query("status == 'fail'")
if not failed_checks.empty:
    display(failed_checks)
    raise ValueError("Hay checks criticos fallidos. Revisar eda_relationship_quality_checks.csv")


Exportado: eda_survey_kpis.csv (1, 6)
Exportado: eda_fan_by_age.csv (4, 4)
Exportado: eda_fan_by_gender.csv (2, 4)
Exportado: eda_movie_views_summary.csv (6, 6)
Exportado: eda_movie_rank_summary.csv (6, 9)
Exportado: eda_movie_opportunities.csv (6, 10)
Exportado: eda_movie_commercial_audience_summary.csv (11, 25)
Exportado: eda_character_opinion_summary.csv (14, 11)
Exportado: eda_quote_character_summary.csv (19, 3)
Exportado: eda_character_merchandising_opportunities.csv (14, 27)
Exportado: eda_universe_overview.csv (6, 2)
Exportado: eda_character_species_summary.csv (27, 2)
Exportado: eda_character_gender_summary.csv (3, 2)
Exportado: eda_character_homeworld_summary.csv (55, 2)
Exportado: eda_planet_business_summary.csv (26, 10)
Exportado: eda_starship_business_summary.csv (56, 13)
Exportado: eda_weapon_business_summary.csv (57, 8)
Exportado: eda_governance_missing_top.csv (45, 4)
Exportado: eda_survey_sample_bias.csv (4, 3)
Exportado: eda_relationship_quality_checks.csv (9, 4)
Expor